# Notebook 01d: Temporal Split EDA
This notebook analyzes the time-based distribution of incident reports. We parse the `Time_Date` field (YYYYMM) and establish chronological boundaries for the train, validation, and test splits to prevent temporal leakage.

In [ ]:
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import seaborn as sns

with open('configs/main_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

df = pd.read_parquet(config['paths']['raw_data'])
print(f"Loaded {len(df)} records.")

## 1. Parse Time_Date

In [ ]:
# Time_Date is YYYYMM integer
df['year'] = df['Time_Date'] // 100
df['month'] = df['Time_Date'] % 100

print(f"Year range: {df['year'].min()} to {df['year'].max()}")
print(f"Month range: {df['month'].min()} to {df['month'].max()}")

## 2. Visualize Reports per Year

In [ ]:
plt.figure(figsize=(12, 6))
df['year'].value_counts().sort_index().plot(kind='bar')
plt.title('Incident Reports per Year')
plt.xlabel('Year')
plt.ylabel('Report Count')
plt.show()

## 3. Define and Validate Split Boundaries

In [ ]:
# Example strategy based on plan: Train 2003-2017, Val 2018, Test 2019-2020
def get_split(year):
    if year < 2018:
        return 'train'
    elif year == 2018:
        return 'val'
    else:
        return 'test'

df['split'] = df['year'].apply(get_split)
split_counts = df['split'].value_counts(normalize=True) * 100
print("Split distribution (%):")
print(split_counts)

print("\nAbsolute counts:")
print(df['split'].value_counts())

## 4. Save Split Assignments

In [ ]:
df[['acn_num_ACN', 'year', 'month', 'split']].to_parquet('data/processed/temporal_splits.parquet')
print("Temporal splits saved.")